# Notebook 02 — Entrenamiento Contrastivo (SupCon)

**Objetivo:** Entrenar el encoder ResNet18 con Supervised Contrastive Learning para producir embeddings de 1024 dims.

**IMPORTANTE:** Este notebook está diseñado para ejecutarse en **Google Colab con GPU**.
En CPU local toma ~2h por epoch con batch_size=32. En Colab T4 toma ~2-5 min por epoch con batch_size=256.

## Cómo ejecutar en Colab
1. Runtime → Change runtime type → **T4 GPU** (gratis) o A100 (Pro)
2. Descomenta las celdas marcadas `# COLAB`
3. Runtime → Run all
4. Al terminar: descarga `artifacts/checkpoints/encoder_best.pt` para continuar localmente

In [ ]:
# COLAB — descomenta estas líneas
# !git clone https://github.com/TU_USUARIO/Malaria-Dectetion-Deeplearning.git
# %cd Malaria-Dectetion-Deeplearning
# !pip install -r requirements.txt -q

# Montar Drive y copiar dataset:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/cell_images' ./cell_images
# # O usar Kaggle API:
# # !pip install kaggle -q
# # !kaggle datasets download -d iarunava/cell-images-for-detecting-malaria
# # !unzip -q cell-images-for-detecting-malaria.zip

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_json
from src.data.split import make_stratified_split
from src.data.augmentations import get_contrastive_transform
from src.data.dataset import SupConPairDataset
from src.models.encoder import ContrastiveEncoder
from src.training.train_contrastive import train
from src.visualization.training_plots import plot_training_curves

import matplotlib.pyplot as plt
%matplotlib inline

set_global_seed(42)

In [ ]:
cfg = load_config('configs/contrastive.yaml')
data_cfg = load_config('configs/data.yaml')
print('Configuración del encoder:', cfg['encoder'])
print('Épocas:', cfg['training']['epochs'])

## 1. Preparar datos

In [ ]:
from torch.utils.data import DataLoader

# Generar splits si no existen
train_df, val_df, test_df = make_stratified_split(
    dataset_root=data_cfg['dataset_root'],
    processed_dir=data_cfg['processed_dir'],
    seed=data_cfg['seed'],
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

use_gpu = torch.cuda.is_available()
aug_cfg = cfg.get('augmentations', {})
transform = get_contrastive_transform(
    img_size=aug_cfg.get('img_size', 96),
    blur_prob=aug_cfg.get('blur_prob', 0.3),
)

batch_size = cfg['training']['batch_size_gpu'] if use_gpu else cfg['training']['batch_size_cpu']
num_workers = 2 if use_gpu else 0
print(f'batch_size={batch_size} | num_workers={num_workers} | GPU={use_gpu}')

train_ds = SupConPairDataset('data/processed/train.csv', transform=transform)
val_ds   = SupConPairDataset('data/processed/val.csv',   transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=use_gpu, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=use_gpu)

## 2. Entrenamiento SupCon

In [ ]:
from pathlib import Path
Path('artifacts/checkpoints').mkdir(parents=True, exist_ok=True)

history = train(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir='artifacts/checkpoints',
)
print('\nEntrenamiento completado!')
print(f'Mejor val_loss: {min(history["val"]):.4f} (epoch {history["val"].index(min(history["val"]))+1})')

In [ ]:
fig = plot_training_curves(history, save_path='artifacts/figures/training_curves.png')
plt.show()

import json
Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
with open('artifacts/logs/contrastive_history.json', 'w') as f:
    json.dump(history, f)

## 3. Descargar checkpoint (solo Colab)

In [ ]:
# COLAB — descomenta para descargar el checkpoint a tu PC
# from google.colab import files
# files.download('artifacts/checkpoints/encoder_best.pt')

# O guardar en Drive:
# import shutil
# shutil.copy('artifacts/checkpoints/encoder_best.pt',
#             '/content/drive/MyDrive/malaria_encoder_best.pt')

## Siguiente paso
Coloca `encoder_best.pt` en `artifacts/checkpoints/` y ejecuta:
```bash
python -m scripts.extract_embeddings --checkpoint artifacts/checkpoints/encoder_best.pt
```
O abre `notebooks/03_extract_embeddings.ipynb`.